In [3]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image:
# https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse)   # Metapackage for data manipulation and visualization

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking Run or pressing Shift+Enter)
# will list all files under the input directory

input_dir <- "../input"

# List all files recursively
all_files <- list.files(
  path = input_dir,
  recursive = TRUE,
  full.names = TRUE
)

# Get directories containing files
dirs <- unique(dirname(all_files))

# Display directory names and up to the first 5 files
for (dir in dirs) {
  files <- list.files(dir, full.names = TRUE)

  if (length(files) > 0) {
    cat("\nDirectory:", dir, "\n")

    # Print up to the first 5 files
    n_to_print <- min(5, length(files))
    cat(paste0("  ", files[1:n_to_print]), sep = "\n")

    # Count and display remaining items if they exceed 5
    remaining <- length(files) - 5
    if (remaining > 0) {
      cat("  ... and", remaining, "more items\n")
    }
  }
}

# You can write up to 20GB to the current directory (/kaggle/working/)
# that gets preserved as output when you create a version using
# "Save & Run All".
# You can also write temporary files to /kaggle/temp/,
# but they won't be saved outside of the current session.

working_dir <- "/kaggle/working"
temp_dir <- "/kaggle/temp"

cat("\nWorking directory:", working_dir, "\n")
cat("Temporary directory:", temp_dir, "\n")

# Kaggle notebooks automatically mount attached datasets under
# "../input/<dataset-name>/"
#
# Example:
# data <- read_csv("../input/your-dataset-name/data.csv")
#
# or, for larger files:
# library(data.table)
# data <- fread("../input/your-dataset-name/data.csv")



Directory: ../input/datasets/mashlyn/online-retail-ii-uci 
  ../input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv

Working directory: /kaggle/working 
Temporary directory: /kaggle/temp 


In [4]:
# LAB 3: MULTI-SOURCE RETAIL SALES DATA INTEGRATION & ANALYSIS
# Complete Single R Script for Kaggle

# 0. INSTALL / LOAD PACKAGES

packages <- c(
  "tidyverse",
  "jsonlite",
  "readxl",
  "writexl",
  "DBI",
  "RSQLite",
  "lubridate"
)

for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(jsonlite)
library(readxl)
library(writexl)
library(DBI)
library(RSQLite)
library(lubridate)

cat("\n==============\n")
cat("LAB 3: RETAIL SALES DATA INTEGRATION AND ANALYSIS\n")
cat("==============\n")


# 1. IMPORT ORIGINAL ONLINE RETAIL II DATASET

cat("\n==================== STEP 1 ====================\n")
cat("IMPORTING ORIGINAL DATASET\n")

uci_file <- "/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv"

if (!file.exists(uci_file)) {
  stop("Dataset not found. Check the Kaggle dataset path.")
}

retail_raw <- read.csv(
  uci_file,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

cat("\nOriginal dataset dimensions:\n")
cat("Rows:", nrow(retail_raw), "\n")
cat("Columns:", ncol(retail_raw), "\n")

cat("\nOriginal column names:\n")
print(names(retail_raw))

cat("\nFirst 6 records:\n")
print(head(retail_raw))


# 2. STANDARDIZE ONLINE RETAIL II COLUMN NAMES

cat("\n==================== STEP 2 ====================\n")
cat("STANDARDIZING COLUMN NAMES\n")

# Remove spaces from column names
names(retail_raw) <- trimws(names(retail_raw))

# Handle Online Retail II naming convention
if ("Invoice" %in% names(retail_raw)) {
  names(retail_raw)[names(retail_raw) == "Invoice"] <- "InvoiceNo"
}

if ("Price" %in% names(retail_raw)) {
  names(retail_raw)[names(retail_raw) == "Price"] <- "UnitPrice"
}

if ("Customer ID" %in% names(retail_raw)) {
  names(retail_raw)[names(retail_raw) == "Customer ID"] <- "CustomerID"
}

cat("\nStandardized column names:\n")
print(names(retail_raw))


# 3. CHECK REQUIRED COLUMNS

required_columns <- c(
  "InvoiceNo",
  "StockCode",
  "Description",
  "Quantity",
  "InvoiceDate",
  "UnitPrice",
  "CustomerID",
  "Country"
)

missing_columns <- setdiff(
  required_columns,
  names(retail_raw)
)

if (length(missing_columns) > 0) {
  stop(
    paste(
      "The following required columns are missing:",
      paste(missing_columns, collapse = ", ")
    )
  )
}


# 4. CREATE THE THREE REQUIRED DATA SOURCES

cat("\n==================== STEP 3 ====================\n")
cat("CREATING CSV, JSON AND EXCEL DATA SOURCES\n")


# 
# 4.1 TRANSACTIONS CSV
# 

transactions <- retail_raw %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

write.csv(
  transactions,
  "/kaggle/working/transactions.csv",
  row.names = FALSE
)


# 
# 4.2 PRODUCTS JSON
# 

products <- retail_raw %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  filter(
    !is.na(StockCode)
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

write_json(
  products,
  "/kaggle/working/products.json",
  pretty = TRUE,
  auto_unbox = TRUE
)


# 
# 4.3 CUSTOMERS EXCEL
# 

customers <- retail_raw %>%
  select(
    CustomerID,
    Country
  ) %>%
  filter(
    !is.na(CustomerID)
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

write_xlsx(
  customers,
  "/kaggle/working/customers.xlsx"
)


cat("\nRequired files successfully created:\n")
cat("1. transactions.csv\n")
cat("2. products.json\n")
cat("3. customers.xlsx\n")


# 5. IMPORT THE THREE DATA SOURCES

cat("\n==================== TASK 1 ====================\n")
cat("IMPORTING CSV, JSON AND EXCEL\n")

transactions <- read.csv(
  "/kaggle/working/transactions.csv",
  stringsAsFactors = FALSE
)

products <- fromJSON(
  "/kaggle/working/products.json"
)

products <- as.data.frame(
  products,
  stringsAsFactors = FALSE
)

customers <- read_excel(
  "/kaggle/working/customers.xlsx"
)

customers <- as.data.frame(
  customers,
  stringsAsFactors = FALSE
)


cat("\nTransactions:\n")
cat(nrow(transactions), "rows x",
    ncol(transactions), "columns\n")

cat("\nProducts:\n")
cat(nrow(products), "rows x",
    ncol(products), "columns\n")

cat("\nCustomers:\n")
cat(nrow(customers), "rows x",
    ncol(customers), "columns\n")


# 6. INSPECT DATA

cat("\n----- DATA STRUCTURE -----\n")

cat("\nTransactions:\n")
str(transactions)

cat("\nProducts:\n")
str(products)

cat("\nCustomers:\n")
str(customers)


# 7. CONVERT DATA TYPES

cat("\n==================== CLEANING ====================\n")

transactions$InvoiceNo <- as.character(
  transactions$InvoiceNo
)

transactions$StockCode <- as.character(
  transactions$StockCode
)

transactions$CustomerID <- as.character(
  transactions$CustomerID
)

customers$CustomerID <- as.character(
  customers$CustomerID
)

products$StockCode <- as.character(
  products$StockCode
)

transactions$Quantity <- suppressWarnings(
  as.numeric(transactions$Quantity)
)

products$UnitPrice <- suppressWarnings(
  as.numeric(products$UnitPrice)
)


# 8. MISSING VALUE ANALYSIS

cat("\n----- MISSING VALUES -----\n")

cat("\nTransactions:\n")
print(colSums(is.na(transactions)))

cat("\nProducts:\n")
print(colSums(is.na(products)))

cat("\nCustomers:\n")
print(colSums(is.na(customers)))


# 9. DUPLICATE ANALYSIS

cat("\n----- DUPLICATES -----\n")

transaction_duplicates <- sum(
  duplicated(transactions)
)

product_duplicates <- sum(
  duplicated(products)
)

customer_duplicates <- sum(
  duplicated(customers)
)

cat("Transaction duplicates:",
    transaction_duplicates, "\n")

cat("Product duplicates:",
    product_duplicates, "\n")

cat("Customer duplicates:",
    customer_duplicates, "\n")


# 10. CLEAN TRANSACTIONS

transaction_rows_before <- nrow(transactions)

transactions <- transactions %>%
  distinct() %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(Quantity),
    Quantity > 0
  )

transaction_rows_after <- nrow(transactions)

cat("\nTransaction rows before cleaning:",
    transaction_rows_before, "\n")

cat("Transaction rows after cleaning:",
    transaction_rows_after, "\n")


# 11. CLEAN PRODUCTS

product_rows_before <- nrow(products)

products <- products %>%
  distinct() %>%
  filter(
    !is.na(StockCode),
    !is.na(UnitPrice),
    UnitPrice > 0
  ) %>%
  group_by(StockCode) %>%
  slice(1) %>%
  ungroup()

product_rows_after <- nrow(products)

cat("\nProduct rows before cleaning:",
    product_rows_before, "\n")

cat("Product rows after cleaning:",
    product_rows_after, "\n")


# 12. CLEAN CUSTOMERS

customers <- customers %>%
  filter(
    !is.na(CustomerID),
    CustomerID != ""
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

cat("\nCleaned customer records:",
    nrow(customers), "\n")


# 13. INTEGRATE TRANSACTIONS + PRODUCTS

cat("\n==================== TASK 2 ====================\n")
cat("DATA INTEGRATION\n")

sales_products <- transactions %>%
  left_join(
    products,
    by = "StockCode"
  )


# Identify unmatched products
unmatched_products <- sales_products %>%
  filter(
    is.na(UnitPrice)
  )

cat("\nUnmatched product records:",
    nrow(unmatched_products), "\n")


# 14. INTEGRATE CUSTOMERS

final_data <- sales_products %>%
  left_join(
    customers,
    by = "CustomerID"
  )


# Identify unmatched customers
unmatched_customers <- final_data %>%
  filter(
    is.na(Country)
  )

cat("Unmatched customer records:",
    nrow(unmatched_customers), "\n")


# 15. REMOVE INVALID UNIT PRICES

final_data <- final_data %>%
  filter(
    !is.na(UnitPrice),
    UnitPrice > 0
  )


# 16. CREATE REVENUE

final_data <- final_data %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )


# 17. FINAL DATASET VERIFICATION

cat("\n----- FINAL DATASET -----\n")

cat("Rows:",
    nrow(final_data), "\n")

cat("Columns:",
    ncol(final_data), "\n")

cat("\nColumns:\n")
print(names(final_data))

cat("\nFirst 10 records:\n")
print(head(final_data, 10))

cat("\nMissing values:\n")
print(colSums(is.na(final_data)))


# 18. CLEANING DECISIONS

cat("\n----- CLEANING DECISIONS -----\n")

cat("
1. Duplicate records were removed.
2. Transactions with missing essential fields were removed.
3. Transactions with Quantity <= 0 were removed.
4. Products with missing, zero or negative UnitPrice were removed.
5. Duplicate StockCode records were reduced to one product record.
6. Customers without CustomerID were removed.
7. Revenue was calculated as Quantity * UnitPrice.
8. LEFT JOIN was used because transactions are the main dataset.
9. LEFT JOIN also allows unmatched products and customers to be identified.
")


# TASK 3: SALES ANALYSIS

cat("\n==================== TASK 3 ====================\n")
cat("SALES AND CUSTOMER ANALYSIS\n")


# 19. TOTAL SALES REVENUE

total_revenue <- final_data %>%
  summarise(
    Total_Revenue = sum(
      Revenue,
      na.rm = TRUE
    )
  )

cat("\n1. TOTAL SALES REVENUE\n")
print(total_revenue)


# 20. TOP 5 PRODUCTS

top5_products <- final_data %>%
  group_by(
    StockCode,
    Description
  ) %>%
  summarise(
    Total_Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    Total_Quantity = sum(
      Quantity,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Total_Revenue)
  ) %>%
  slice_head(n = 5)

cat("\n2. TOP 5 PRODUCTS BY REVENUE\n")
print(top5_products)


# 21. TOP 5 COUNTRIES

top5_countries <- final_data %>%
  filter(
    !is.na(Country)
  ) %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    Total_Quantity = sum(
      Quantity,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Total_Revenue)
  ) %>%
  slice_head(n = 5)

cat("\n3. TOP 5 COUNTRIES BY REVENUE\n")
print(top5_countries)


# 22. TOP 5 CUSTOMERS

top5_customers <- final_data %>%
  filter(
    !is.na(CustomerID)
  ) %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(
      Revenue,
      na.rm = TRUE
    ),
    Total_Orders = n_distinct(
      InvoiceNo
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Total_Purchase_Value)
  ) %>%
  slice_head(n = 5)

cat("\n4. TOP 5 CUSTOMERS BY PURCHASE VALUE\n")
print(top5_customers)


# 23. CUSTOMER VALUE CLASSIFICATION

customer_values <- final_data %>%
  filter(
    !is.na(CustomerID)
  ) %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  )


# Calculate quartile thresholds
q1 <- quantile(
  customer_values$Total_Purchase_Value,
  0.25,
  na.rm = TRUE
)

q2 <- quantile(
  customer_values$Total_Purchase_Value,
  0.50,
  na.rm = TRUE
)

q3 <- quantile(
  customer_values$Total_Purchase_Value,
  0.75,
  na.rm = TRUE
)

cat("\nCustomer value thresholds:\n")
cat("25th percentile:", round(q1, 2), "\n")
cat("50th percentile:", round(q2, 2), "\n")
cat("75th percentile:", round(q3, 2), "\n")


# 24. CASE_WHEN CUSTOMER CLASSIFICATION

customer_values <- customer_values %>%
  mutate(
    Customer_Value_Segment = case_when(
      
      Total_Purchase_Value <= q1 ~
        "Low Value",
      
      Total_Purchase_Value <= q2 ~
        "Medium Value",
      
      Total_Purchase_Value <= q3 ~
        "High Value",
      
      Total_Purchase_Value > q3 ~
        "Premium",
      
      TRUE ~ "Unknown"
    )
  )

cat("\nCustomer value segments:\n")

segment_summary <- customer_values %>%
  count(
    Customer_Value_Segment
  ) %>%
  arrange(
    desc(n)
  )

print(segment_summary)


# 25. ADD SEGMENT TO FINAL DATASET

final_data <- final_data %>%
  left_join(
    customer_values %>%
      select(
        CustomerID,
        Customer_Value_Segment
      ),
    by = "CustomerID"
  )


# 26. COUNTRY PERFORMANCE

country_performance <- final_data %>%
  filter(
    !is.na(Country)
  ) %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    Total_Quantity = sum(
      Quantity,
      na.rm = TRUE
    ),
    Number_of_Customers = n_distinct(
      CustomerID
    ),
    Number_of_Invoices = n_distinct(
      InvoiceNo
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Total_Revenue)
  )

cat("\n----- MARKET PERFORMANCE -----\n")

cat("\nTop 10 countries:\n")
print(
  slice_head(
    country_performance,
    n = 10
  )
)


# 27. HIGH AND UNDERPERFORMING MARKET

# Exclude United Kingdom so the comparison is more useful
# for identifying another international market.

non_uk_markets <- country_performance %>%
  filter(
    toupper(Country) != "UNITED KINGDOM"
  )

if (nrow(non_uk_markets) > 0) {
  
  high_market <- non_uk_markets %>%
    slice_max(
      Total_Revenue,
      n = 1
    )
  
  low_market <- non_uk_markets %>%
    slice_min(
      Total_Revenue,
      n = 1
    )
  
} else {
  
  high_market <- country_performance %>%
    slice_max(
      Total_Revenue,
      n = 1
    )
  
  low_market <- country_performance %>%
    slice_min(
      Total_Revenue,
      n = 1
    )
}


cat("\nHigh-performing market:\n")
print(high_market)

cat("\nUnderperforming market:\n")
print(low_market)


# 28. SQLITE DATABASE

cat("\n==================== TASK 4 ====================\n")
cat("SQLITE DATABASE\n")

database_file <- "/kaggle/working/retail_sales.db"

if (file.exists(database_file)) {
  file.remove(database_file)
}

con <- dbConnect(
  SQLite(),
  database_file
)

cat("\nSQLite database created.\n")


# 29. WRITE FINAL DATASET TO SQLITE

dbWriteTable(
  con,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

cat("Table 'retail_sales' created.\n")


# 30. CHECK DATABASE

cat("\nDatabase tables:\n")
print(
  dbListTables(con)
)

cat("\nRecords in retail_sales:\n")

db_count <- dbGetQuery(
  con,
  "SELECT COUNT(*) AS Number_of_Records
   FROM retail_sales"
)

print(db_count)


# 31. SQL QUERY 1 - TOP 5 CUSTOMERS

cat("\n----- SQL QUERY 1 -----\n")
cat("TOP 5 CUSTOMERS BY REVENUE\n")

query1 <- "
SELECT
    CustomerID,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM retail_sales
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5;
"

sql_top_customers <- dbGetQuery(
  con,
  query1
)

print(sql_top_customers)


# 32. SQL QUERY 2 - REVENUE BY COUNTRY

cat("\n----- SQL QUERY 2 -----\n")
cat("TOTAL REVENUE BY COUNTRY\n")

query2 <- "
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM retail_sales
WHERE Country IS NOT NULL
GROUP BY Country
ORDER BY Total_Revenue DESC
LIMIT 10;
"

sql_country_revenue <- dbGetQuery(
  con,
  query2
)

print(sql_country_revenue)


# 33. SQL QUERY 3 - TOTAL REVENUE

cat("\n----- SQL QUERY 3 -----\n")

query3 <- "
SELECT
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM retail_sales;
"

sql_total_revenue <- dbGetQuery(
  con,
  query3
)

print(sql_total_revenue)


# 34. CLOSE DATABASE

dbDisconnect(con)

cat("\nSQLite connection closed.\n")


# 35. SAVE FINAL OUTPUT FILES

cat("\n==================== SAVING OUTPUTS ====================\n")

write.csv(
  final_data,
  "/kaggle/working/final_retail_sales.csv",
  row.names = FALSE
)

write.csv(
  top5_products,
  "/kaggle/working/top5_products.csv",
  row.names = FALSE
)

write.csv(
  top5_countries,
  "/kaggle/working/top5_countries.csv",
  row.names = FALSE
)

write.csv(
  top5_customers,
  "/kaggle/working/top5_customers.csv",
  row.names = FALSE
)

write.csv(
  country_performance,
  "/kaggle/working/country_performance.csv",
  row.names = FALSE
)

write.csv(
  customer_values,
  "/kaggle/working/customer_value_segments.csv",
  row.names = FALSE
)

write.csv(
  segment_summary,
  "/kaggle/working/customer_segment_summary.csv",
  row.names = FALSE
)


# 36. THREE BUSINESS INSIGHTS

cat("\n==============\n")
cat("THREE IMPORTANT BUSINESS INSIGHTS\n")
cat("==============\n")


# Insight 1
cat("\nINSIGHT 1 - PRODUCT PERFORMANCE\n")

if (nrow(top5_products) > 0) {
  
  cat(
    paste0(
      "The top-performing product is '",
      as.character(top5_products$Description[1]),
      "' with revenue of ",
      round(
        top5_products$Total_Revenue[1],
        2
      ),
      ". High-revenue products should receive priority ",
      "for inventory planning and promotion.\n"
    )
  )
}


# Insight 2
cat("\nINSIGHT 2 - MARKET PERFORMANCE\n")

if (nrow(top5_countries) > 0) {
  
  cat(
    paste0(
      top5_countries$Country[1],
      " is the highest-revenue market with total revenue of ",
      round(
        top5_countries$Total_Revenue[1],
        2
      ),
      ". This indicates strong demand and makes the market ",
      "important for continued business investment.\n"
    )
  )
}


# Insight 3
cat("\nINSIGHT 3 - CUSTOMER VALUE\n")

if (nrow(top5_customers) > 0) {
  
  premium_customers <- sum(
    customer_values$Customer_Value_Segment == "Premium",
    na.rm = TRUE
  )
  
  cat(
    paste0(
      "The highest-value customer generated ",
      round(
        top5_customers$Total_Purchase_Value[1],
        2
      ),
      " in purchases. There are ",
      premium_customers,
      " Premium customers. These customers should be ",
      "targeted with loyalty programs and personalized offers.\n"
    )
  )
}


# 37. FINAL SUMMARY

cat("\n==============\n")
cat("LAB COMPLETED SUCCESSFULLY\n")
cat("==============\n")

cat("\nFinal dataset:\n")
cat("Rows:", nrow(final_data), "\n")
cat("Columns:", ncol(final_data), "\n")

cat(
  "\nTotal Revenue:",
  round(
    sum(final_data$Revenue, na.rm = TRUE),
    2
  ),
  "\n"
)

cat("\nFiles generated in /kaggle/working:\n")

print(
  list.files(
    "/kaggle/working",
    pattern = "\\.(csv|json|xlsx|db)$"
  )
)

cat("\n==============\n")
cat("END OF LAB 3\n")
cat("==============\n")



LAB 3: RETAIL SALES DATA INTEGRATION AND ANALYSIS

==================== STEP 1 ====================
IMPORTING ORIGINAL DATASET

Original dataset dimensions:
Rows: 1067371 
Columns: 8 

Original column names:
[1] "Invoice"     "StockCode"   "Description" "Quantity"    "InvoiceDate"
[6] "Price"       "Customer ID" "Country"    

First 6 records:
  Invoice StockCode                         Description Quantity
1  489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS       12
2  489434    79323P                  PINK CHERRY LIGHTS       12
3  489434    79323W                 WHITE CHERRY LIGHTS       12
4  489434     22041        RECORD FRAME 7" SINGLE SIZE        48
5  489434     21232      STRAWBERRY CERAMIC TRINKET BOX       24
6  489434     22064          PINK DOUGHNUT TRINKET POT        24
          InvoiceDate Price Customer ID        Country
1 2009-12-01 07:45:00  6.95       13085 United Kingdom
2 2009-12-01 07:45:00  6.75       13085 United Kingdom
3 2009-12-01 07:45:00  6.75      